# 09. SacreBLEU, chrF++, & COMET Benchmark Evaluation

**Requires GPU + Hub access.** This notebook downloads/loads 8B-parameter models (Aya-23-8B, and/or Llama-3.1-8B which additionally requires accepting Meta's license on the HuggingFace Hub). Run on Colab with a GPU runtime -- see the setup cell below, which auto-clones the repo when a Colab GPU is detected.

Evaluates one model/checkpoint against the fixed `master_test.csv` split and saves results for the ablation study (notebook 11).

In [ ]:
# ============================================================
# PATH BOOSTER — Guarantees project root in sys.path & CWD
# ============================================================
import os, sys
try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)
proj_dir = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


In [ ]:
import os

if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
    if not os.path.exists("Ekegusii-LLM-Translation"):
        os.system("git clone https://github.com/aykahsay/Ekegusii-LLM-Translation.git")
    os.chdir("Ekegusii-LLM-Translation")
    os.system("pip install -q -r requirements.txt")
elif not os.path.exists("src") and os.path.basename(os.getcwd()) == "notebooks":
    # Running locally via `jupyter nbconvert` from within notebooks/ --
    # the repo root (containing src/, data/) is one directory up.
    os.chdir("..")

import sys
sys.path.insert(0, os.getcwd())

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


In [ ]:
MODEL_NAME = 'aya'              # 'aya' or 'llama'
EXPERIMENT_ID = 'E4_Trilingual'  # must match a completed training run, or None for zero-shot (E0)
ADAPTER_PATH = f'checkpoints/{MODEL_NAME}/{EXPERIMENT_ID}/best' if EXPERIMENT_ID else None

In [ ]:
import os, sys
p = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p); os.chdir(p)

from src.cli.evaluate import run_evaluate

results = run_evaluate(MODEL_NAME, source_lang='English', target_lang='Ekegusii', adapter_path=ADAPTER_PATH)
results

## Save results for the ablation study

In [ ]:
import json
from pathlib import Path

out_dir = Path('experiments') / (EXPERIMENT_ID or 'E0_Baseline')
out_dir.mkdir(parents=True, exist_ok=True)
results_path = out_dir / 'results.json'

existing = json.loads(results_path.read_text()) if results_path.exists() else {'experiment_id': EXPERIMENT_ID}
existing[MODEL_NAME] = results
results_path.write_text(json.dumps(existing, indent=2))
print(f'Saved to {results_path}')

## COMET (optional, slower -- downloads a ~1.7GB checkpoint on first use)

In [ ]:
import os, sys
p = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p); os.chdir(p)

from src.experiments.base import BaseExperiment
from src.master_corpus.manager import MasterCorpusManager
from src.evaluation.comet import CometEvaluator
from src.models.aya.inference import translate_with_aya
from src.models.llama.inference import translate_with_llama

class _EvalHelper(BaseExperiment):
    experiment_id = 'notebook09'
    def build_training_tasks(self): raise NotImplementedError

helper = _EvalHelper(MasterCorpusManager())
test_pairs = helper.build_test_pairs('English', 'Ekegusii')
sources, references = test_pairs['source'].tolist(), test_pairs['target'].tolist()

translate_fn = translate_with_aya if MODEL_NAME == 'aya' else translate_with_llama
predictions = translate_fn(sources, 'English', 'Ekegusii', adapter_path=ADAPTER_PATH)

comet_result = CometEvaluator().compute(predictions, references, sources)
print(f"Mean COMET: {comet_result['mean_score']:.4f}")